In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
from xgboost import XGBRegressor, plot_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP (pre-installed on Kaggle)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available – skipping SHAP plots")

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2e3250",
    "axes.labelcolor":  "#c8cfe8",
    "xtick.color":      "#8892b0",
    "ytick.color":      "#8892b0",
    "text.color":       "#c8cfe8",
    "grid.color":       "#1e2340",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
    "figure.dpi":       120,
})
ACCENT   = "#64ffda"
ACCENT2  = "#f472b6"
ACCENT3  = "#fbbf24"
BLUE     = "#38bdf8"
PURPLE   = "#818cf8"

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/bapdesilva/betel-ds/betel_yield_dataset_40000_v2.csv")

print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape         : {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Target range  : {df['Leaf_Count_per_Bed'].min():.0f} – {df['Leaf_Count_per_Bed'].max():.0f}")
print(f"  Target mean   : {df['Leaf_Count_per_Bed'].mean():.2f} ± {df['Leaf_Count_per_Bed'].std():.2f}")
print("=" * 55)
print(df.describe().round(2))


# ── 2. Feature engineering ────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # Interaction terms (domain-informed)
    df["Moisture_x_Rain"]    = df["Soil_Moisture"] * df["Rainfall"]
    df["Moisture_x_Humid"]   = df["Soil_Moisture"] * df["Humidity"]
    df["NPK_sum"]            = df["N"] + df["P"] + df["K"]
    df["NPK_product"]        = df["N"] * df["P"] * df["K"]
    df["Temp_x_Humid"]       = df["Temperature"] * df["Humidity"]
    df["Rain_x_Humid"]       = df["Rainfall"] * df["Humidity"]
    df["pH_x_NPK"]           = df["Soil_pH"] * df["NPK_sum"]
    df["Moisture_sq"]        = df["Soil_Moisture"] ** 2
    df["Rain_sq"]            = df["Rainfall"] ** 2

    # Log transforms (reduces skew on rain)
    df["log_Rain"]           = np.log1p(df["Rainfall"])
    df["log_NPK"]            = np.log1p(df["NPK_sum"])

    # Ratios
    df["N_to_P"]             = df["N"] / (df["P"] + 1e-6)
    df["Moisture_to_Rain"]   = df["Soil_Moisture"] / (df["Rainfall"] + 1e-6)

    # Optimal range flags (domain heuristics for betel)
    df["optimal_moisture"]   = ((df["Soil_Moisture"] >= 55) & (df["Soil_Moisture"] <= 75)).astype(int)
    df["optimal_temp"]       = ((df["Temperature"]   >= 25) & (df["Temperature"]   <= 32)).astype(int)
    df["optimal_pH"]         = ((df["Soil_pH"]       >= 5.5) & (df["Soil_pH"]      <= 7.0)).astype(int)
    df["optimal_rain"]       = ((df["Rainfall"]      >= 150) & (df["Rainfall"]     <= 300)).astype(int)

    return df

df_eng = engineer_features(df)

FEATURES = [c for c in df_eng.columns if c != "Leaf_Count_per_Bed"]
TARGET   = "Leaf_Count_per_Bed"

X = df_eng[FEATURES]
y = df_eng[TARGET]

print(f"\n  Total features after engineering: {len(FEATURES)}")

# ── 3. Train / validation / test split ───────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=42  # ~10% of total
)
print(f"\n  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ── 4. Optuna hyperparameter search ──────────────────────────
print("\n  Running Optuna hyperparameter search (120 trials)…")

def objective(trial):
    params = {
        "n_estimators":       trial.suggest_int("n_estimators", 400, 1200),
        "max_depth":          trial.suggest_int("max_depth", 3, 9),
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample":          trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel":  trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_child_weight":   trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma":              trial.suggest_float("gamma", 0.0, 5.0),
        "random_state":       42,
        "tree_method":        "hist",
        "device":             "cuda",    # uses GPU on Kaggle – falls back to cpu
        "eval_metric":        "rmse",
        "early_stopping_rounds": 40,
        "verbosity":          0,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    preds = model.predict(X_val)
    return r2_score(y_val, preds)

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=120, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    "random_state": 42,
    "tree_method":  "hist",
    "device":       "cuda",
    "eval_metric":  ["rmse", "mae"],
    "early_stopping_rounds": 50,
    "verbosity":    0,
})
print(f"\n  Best trial R²  : {study.best_value:.6f}")
print(f"  Best params    : {best_params}")

# ── 5. Train final model with learning curves ─────────────────
print("\n  Training final model…")

model = XGBRegressor(**best_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)

evals = model.evals_result()
train_rmse = evals["validation_0"]["rmse"]
val_rmse   = evals["validation_1"]["rmse"]
train_mae  = evals["validation_0"]["mae"]
val_mae    = evals["validation_1"]["mae"]

# Predictions
y_train_pred = model.predict(X_train)
y_val_pred   = model.predict(X_val)
y_test_pred  = model.predict(X_test)

def regression_report(y_true, y_pred, split_name):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"\n  ── {split_name} ──")
    print(f"    R²   : {r2:.6f}  ({r2*100:.2f}%)")
    print(f"    MAE  : {mae:.4f} leaves")
    print(f"    RMSE : {rmse:.4f} leaves")
    print(f"    MAPE : {mape:.2f}%")
    return {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape}

print("\n" + "=" * 55)
print("  FINAL MODEL PERFORMANCE")
print("=" * 55)
tr_metrics  = regression_report(y_train, y_train_pred, "Train")
val_metrics = regression_report(y_val,   y_val_pred,   "Validation")
te_metrics  = regression_report(y_test,  y_test_pred,  "Test")

# ── 6. Plot 1: Learning curves (RMSE & MAE over rounds) ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Learning Curves — XGBoost Boosting Rounds", fontsize=14, color=ACCENT, y=1.02)

rounds = range(1, len(train_rmse) + 1)
best_r = model.best_iteration

for ax, (tr, vl, metric) in zip(axes, [
    (train_rmse, val_rmse, "RMSE"),
    (train_mae,  val_mae,  "MAE"),
]):
    ax.plot(rounds, tr, color=ACCENT,  lw=1.5, label="Train",      alpha=0.9)
    ax.plot(rounds, vl, color=ACCENT2, lw=1.5, label="Validation", alpha=0.9)
    ax.axvline(best_r, color=ACCENT3, ls="--", lw=1.2, label=f"Best round ({best_r})")
    ax.fill_between(rounds, tr, vl, alpha=0.05, color=ACCENT2)
    ax.set_xlabel("Boosting Rounds", fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f"{metric} vs Rounds", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("learning_curves.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: learning_curves.png")


# ── 7. Plot 2: R² "accuracy" curve over rounds ───────────────
# Compute R² at each round using staged predictions
print("\n  Computing R² per round (this may take ~1 min)…")
train_r2_per_round = []
val_r2_per_round   = []

for i in range(0, model.best_iteration + 1, max(1, model.best_iteration // 200)):
    tr_pred_i  = model.predict(X_train, iteration_range=(0, i + 1))
    val_pred_i = model.predict(X_val,   iteration_range=(0, i + 1))
    train_r2_per_round.append((i + 1, r2_score(y_train, tr_pred_i)))
    val_r2_per_round.append((  i + 1, r2_score(y_val,   val_pred_i)))

rnd_tr,  r2_tr  = zip(*train_r2_per_round)
rnd_val, r2_val = zip(*val_r2_per_round)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rnd_tr,  r2_tr,  color=ACCENT,  lw=2,   label="Train R²",      alpha=0.9)
ax.plot(rnd_val, r2_val, color=ACCENT2, lw=2,   label="Validation R²", alpha=0.9)
ax.axhline(0.95, color=ACCENT3, ls="--", lw=1.2, label="0.95 target")
ax.axvline(model.best_iteration, color=BLUE, ls=":", lw=1.2,
           label=f"Best round ({model.best_iteration})")
ax.fill_between(rnd_val, r2_val, 0.95, where=[v >= 0.95 for v in r2_val],
                alpha=0.12, color=ACCENT, label="Above 0.95 target")
ax.set_xlabel("Boosting Rounds", fontsize=12)
ax.set_ylabel("R² Score", fontsize=12)
ax.set_title("R² (Accuracy) Curve — Train vs Validation", fontsize=13, color=ACCENT)
ax.set_ylim(0, 1.01)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("r2_accuracy_curve.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: r2_accuracy_curve.png")

# ── 8. Plot 3: Optuna optimization history ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Optuna Hyperparameter Search", fontsize=13, color=ACCENT)

vals  = [t.value for t in study.trials]
best_ = np.maximum.accumulate(vals)

axes[0].scatter(range(len(vals)), vals, color=ACCENT, s=12, alpha=0.5, label="Trial R²")
axes[0].plot(range(len(best_)), best_, color=ACCENT2, lw=2, label="Best so far")
axes[0].set_xlabel("Trial #"); axes[0].set_ylabel("R² (val)")
axes[0].set_title("Optimization History"); axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Param importance
try:
    param_imp = optuna.importance.get_param_importances(study)
    params_sorted = list(param_imp.keys())[:10]
    imp_vals      = [param_imp[k] for k in params_sorted]
    axes[1].barh(params_sorted[::-1], imp_vals[::-1],
                 color=[ACCENT, ACCENT2, BLUE, PURPLE, ACCENT3]*3)
    axes[1].set_xlabel("Importance"); axes[1].set_title("Hyperparameter Importance")
    axes[1].grid(True, alpha=0.3, axis="x")
except Exception:
    axes[1].text(0.5, 0.5, "Param importance\nnot available",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig("optuna_search.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: optuna_search.png")


# ── 9. Plot 4: Actual vs Predicted ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Actual vs Predicted — Leaf Count per Bed", fontsize=13, color=ACCENT)

for ax, (yt, yp, name, color) in zip(axes, [
    (y_train, y_train_pred, "Train",      ACCENT),
    (y_val,   y_val_pred,   "Validation", ACCENT2),
    (y_test,  y_test_pred,  "Test",       BLUE),
]):
    r2 = r2_score(yt, yp)
    ax.scatter(yt, yp, alpha=0.25, s=8, color=color, label=f"R²={r2:.4f}")
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=1.2, alpha=0.7, label="Perfect fit")
    ax.set_xlabel("Actual", fontsize=11); ax.set_ylabel("Predicted", fontsize=11)
    ax.set_title(f"{name}  R²={r2*100:.2f}%", fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("actual_vs_predicted.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: actual_vs_predicted.png")

# ── 10. Plot 5: Residuals ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Residual Analysis", fontsize=13, color=ACCENT)

for col, (yt, yp, name, color) in enumerate([
    (y_train, y_train_pred, "Train",      ACCENT),
    (y_val,   y_val_pred,   "Validation", ACCENT2),
    (y_test,  y_test_pred,  "Test",       BLUE),
]):
    res = yt - yp

    # Row 0: Residuals vs Predicted
    axes[0, col].scatter(yp, res, alpha=0.25, s=8, color=color)
    axes[0, col].axhline(0, color="white", lw=1, ls="--", alpha=0.6)
    axes[0, col].set_xlabel("Predicted"); axes[0, col].set_ylabel("Residual")
    axes[0, col].set_title(f"{name} — Residuals vs Predicted"); axes[0, col].grid(True, alpha=0.3)

    # Row 1: Residual distribution
    axes[1, col].hist(res, bins=50, color=color, alpha=0.7, edgecolor="none")
    axes[1, col].axvline(0, color="white", lw=1.2, ls="--")
    axes[1, col].axvline(res.mean(), color=ACCENT3, lw=1.2, label=f"Mean={res.mean():.2f}")
    axes[1, col].set_xlabel("Residual"); axes[1, col].set_ylabel("Count")
    axes[1, col].set_title(f"{name} — Residual Distribution"); axes[1, col].legend(fontsize=9)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("residuals.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: residuals.png")


# ── 11. Plot 6: Feature importance (3 methods) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle("Feature Importance — Three Perspectives", fontsize=13, color=ACCENT)

# Method A: XGBoost built-in (weight)
fi_weight = pd.Series(model.get_booster().get_fscore(), name="weight").sort_values(ascending=False).head(15)
axes[0].barh(fi_weight.index[::-1], fi_weight.values[::-1], color=ACCENT, alpha=0.85)
axes[0].set_title("XGBoost Weight"); axes[0].set_xlabel("F-score"); axes[0].grid(True, alpha=0.3, axis="x")

# Method B: Gain-based
fi_gain = pd.Series(
    model.get_booster().get_score(importance_type="gain"), name="gain"
).sort_values(ascending=False).head(15)
axes[1].barh(fi_gain.index[::-1], fi_gain.values[::-1], color=ACCENT2, alpha=0.85)
axes[1].set_title("XGBoost Gain"); axes[1].set_xlabel("Mean Gain"); axes[1].grid(True, alpha=0.3, axis="x")

# Method C: sklearn feature_importances_ (cover)
fi_cover = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(15)
axes[2].barh(fi_cover.index[::-1], fi_cover.values[::-1], color=BLUE, alpha=0.85)
axes[2].set_title("Cover (sklearn)"); axes[2].set_xlabel("Importance"); axes[2].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("feature_importance.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: feature_importance.png")

# ── 12. Plot 7: SHAP values ───────────────────────────────────
if SHAP_AVAILABLE:
    print("\n  Computing SHAP values…")
    explainer  = shap.TreeExplainer(model)
    shap_vals  = explainer.shap_values(X_test)

    # Summary (beeswarm)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_test, feature_names=FEATURES,
                      show=False, max_display=15, color_bar=True)
    plt.title("SHAP Summary — Beeswarm", fontsize=13, color=ACCENT, pad=12)
    plt.tight_layout()
    plt.savefig("shap_beeswarm.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    print("  → Saved: shap_beeswarm.png")

    # Bar summary
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_test, feature_names=FEATURES,
                      plot_type="bar", show=False, max_display=15)
    plt.title("SHAP Mean |Impact| — Feature Ranking", fontsize=13, color=ACCENT, pad=12)
    plt.tight_layout()
    plt.savefig("shap_bar.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    print("  → Saved: shap_bar.png")

    # Dependence plot for top feature
    top_feat = fi_cover.index[0]
    plt.figure(figsize=(10, 5))
    shap.dependence_plot(top_feat, shap_vals, X_test,
                         feature_names=FEATURES, show=False, dot_size=8, alpha=0.5)
    plt.title(f"SHAP Dependence — {top_feat}", fontsize=12, color=ACCENT)
    plt.tight_layout()
    plt.savefig("shap_dependence.png", bbox_inches="tight", facecolor="#0f1117")
    plt.show()
    print("  → Saved: shap_dependence.png")
else:
    print("  ⚠ SHAP not installed — run: !pip install shap")


# ── 13. Plot 8: Error distribution & metrics dashboard ───────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.patch.set_facecolor("#0f1117")
fig.suptitle("Model Performance Dashboard", fontsize=14, color=ACCENT, y=1.01)

# Metrics table
ax0 = fig.add_subplot(gs[0, 0])
ax0.axis("off")
rows = [
    ["Metric", "Train", "Val", "Test"],
    ["R²",     f"{tr_metrics['R2']:.4f}",  f"{val_metrics['R2']:.4f}",  f"{te_metrics['R2']:.4f}"],
    ["MAE",    f"{tr_metrics['MAE']:.3f}", f"{val_metrics['MAE']:.3f}", f"{te_metrics['MAE']:.3f}"],
    ["RMSE",   f"{tr_metrics['RMSE']:.3f}",f"{val_metrics['RMSE']:.3f}",f"{te_metrics['RMSE']:.3f}"],
    ["MAPE%",  f"{tr_metrics['MAPE']:.2f}",f"{val_metrics['MAPE']:.2f}",f"{te_metrics['MAPE']:.2f}"],
]
tbl = ax0.table(cellText=rows[1:], colLabels=rows[0], cellLoc="center", loc="center",
                bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor("#1a1d27" if r % 2 == 0 else "#0f1117")
    cell.set_edgecolor("#2e3250")
    cell.set_text_props(color="#c8cfe8")
    if r == 0:
        cell.set_facecolor("#2e3250")
        cell.set_text_props(color=ACCENT, fontweight="bold")
ax0.set_title("Metrics Summary", color=ACCENT, pad=8)

# R² bar comparison
ax1 = fig.add_subplot(gs[0, 1])
splits = ["Train", "Validation", "Test"]
r2vals = [tr_metrics["R2"], val_metrics["R2"], te_metrics["R2"]]
bars   = ax1.bar(splits, [v * 100 for v in r2vals],
                 color=[ACCENT, ACCENT2, BLUE], edgecolor="#2e3250", linewidth=0.5)
ax1.axhline(95, color=ACCENT3, ls="--", lw=1.2, label="95% target")
for bar, v in zip(bars, r2vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{v*100:.2f}%", ha="center", va="bottom", fontsize=10, color="white")
ax1.set_ylim(0, 105); ax1.set_ylabel("R² (%)"); ax1.legend(fontsize=9)
ax1.set_title("R² by Split"); ax1.grid(True, alpha=0.3, axis="y")

# Test absolute error histogram
ax2 = fig.add_subplot(gs[0, 2])
abs_err = np.abs(y_test - y_test_pred)
ax2.hist(abs_err, bins=40, color=BLUE, alpha=0.8, edgecolor="none")
ax2.axvline(abs_err.mean(), color=ACCENT3, lw=1.5, label=f"Mean={abs_err.mean():.2f}")
ax2.axvline(np.percentile(abs_err, 90), color=ACCENT2, lw=1.5,
            label=f"90th pct={np.percentile(abs_err, 90):.2f}")
ax2.set_xlabel("Absolute Error (leaves)"); ax2.set_ylabel("Count")
ax2.set_title("Test Absolute Error Dist."); ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Predicted distribution vs actual
ax3 = fig.add_subplot(gs[1, 0:2])
ax3.hist(y_test,       bins=50, alpha=0.55, color=ACCENT,  label="Actual",    edgecolor="none")
ax3.hist(y_test_pred,  bins=50, alpha=0.55, color=ACCENT2, label="Predicted", edgecolor="none")
ax3.set_xlabel("Leaf Count per Bed"); ax3.set_ylabel("Count")
ax3.set_title("Distribution: Actual vs Predicted (Test)"); ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Error pct within tolerance
ax4 = fig.add_subplot(gs[1, 2])
tolerances = [1, 2, 3, 4, 5, 7, 10]
within = [np.mean(abs_err <= t) * 100 for t in tolerances]
ax4.plot(tolerances, within, color=ACCENT, lw=2, marker="o", markersize=5)
ax4.axhline(95, color=ACCENT3, ls="--", lw=1.2, label="95% threshold")
ax4.fill_between(tolerances, within, alpha=0.15, color=ACCENT)
ax4.set_xlabel("Tolerance (leaves)"); ax4.set_ylabel("% Predictions Within")
ax4.set_title("Prediction Tolerance Curve"); ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
for t, w in zip(tolerances, within):
    ax4.annotate(f"{w:.0f}%", (t, w), textcoords="offset points",
                 xytext=(0, 6), ha="center", fontsize=8, color="#c8cfe8")

plt.savefig("performance_dashboard.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: performance_dashboard.png")

# ── 14. Plot 9: Cross-validation boxplot ─────────────────────
print("\n  Running 5-fold cross-validation…")
cv_model = XGBRegressor(**{k: v for k, v in best_params.items()
                           if k not in ["eval_metric", "early_stopping_rounds"]})
cv_scores = cross_val_score(cv_model, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=42),
                             scoring="r2", n_jobs=-1)
print(f"  CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")

fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot(cv_scores, vert=False, patch_artist=True, widths=0.4,
                boxprops=dict(facecolor=ACCENT, alpha=0.6, color=ACCENT),
                whiskerprops=dict(color=ACCENT), medianprops=dict(color="white", lw=2),
                flierprops=dict(markerfacecolor=ACCENT2, markersize=6),
                capprops=dict(color=ACCENT))
ax.scatter(cv_scores, [1]*5, color=ACCENT2, s=60, zorder=5, label="Fold scores")
ax.axvline(cv_scores.mean(), color=ACCENT3, ls="--", lw=1.5,
           label=f"Mean = {cv_scores.mean():.4f}")
ax.set_yticks([]); ax.set_xlabel("R² Score")
ax.set_title("5-Fold Cross-Validation R²", color=ACCENT); ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("cross_validation.png", bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("  → Saved: cross_validation.png")